**Human Movement Intelligence | Biomechanical Load Estimation | Presented by: Reza Saadatyar (2026)**

**05 | Classical Regression Baselines and Participant-Independent Validation**

![Verified Essex paired-data regression workflow, training-only GroupKFold and protected held-out assessment](../../images/05_consolidated_workflow.svg)

**Research question:** How accurately can transparent, pre-specified classical models predict the original eleven-position first and third principal femoral-neck FE strain curves from 42-channel wearable IMU trials, when the validation participants have not contributed to model fitting?

**Data foundation:** The original [Essex Figshare v13 dataset](https://doi.org/10.6084/m9.figshare.30225739.v13), with the verified participant/trial registry and source-exact time alignment recorded in [Notebook 04](04_governed_imu_fe_dataset_and_participant_splits.ipynb). Source inputs: 1,729 eligible trials, IMU (101×42), original FE targets (11×2); 168 original FE-empty trials are excluded.

**Study boundary:** Use the frozen Training (29 participants, 1,354 trials) and Validation (4 participants, 192 trials) split. The four reserved Test participants (183 trials) are never used for model fitting, CV, diagnostics, model choice or performance reporting in this notebook. Validation is diagnostic; **model selection is performed using Training-only grouped CV**.

**Target and statistical interpretation:** Published `e1` and `e3` are **unitless FE-model outputs**, not directly measured patient strain. Report original source-grid pointwise and peak errors, with each complete trial treated as one learning sample. Do not interpolate FE values, convert units, claim prospective clinical utility or compare these results with an independent external cohort.

**Execution status:** New research implementation. All model scores and the Stage 05 Quality Gate are **pending genuine Colab execution**; no performance result is asserted in advance.


**00. Colab Repository Setup**

Open an up-to-date, non-destructive checkout before importing research modules. The shared sync utility preserves local changes and ignored datasets.


In [ ]:
# ======================== 00. Colab Repository Setup =========================

from pathlib import Path
import subprocess
import sys

repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_branch = "main"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")

if not repo_path.exists():
    subprocess.run(["git", "clone", "--branch", repo_branch,
                    repo_url, str(repo_path)], check=True)
elif (
    not (repo_path / "python_modules/core/repository_sync.py").is_file()
    or "def sync_repository(" not in (
        repo_path / "python_modules/core/repository_sync.py"
    ).read_text(encoding="utf-8")
):
    cmd = ["git", "-C", str(repo_path)]
    origin = subprocess.check_output(cmd + ["remote", "get-url", "origin"],
                                     text=True).strip()
    if origin.rstrip("/").removesuffix(".git") != repo_url.removesuffix(".git"):
        raise RuntimeError("Existing repository has an unexpected origin.")
    branch = subprocess.check_output(cmd + ["branch", "--show-current"],
                                     text=True).strip()
    dirty = subprocess.check_output(cmd + ["status", "--porcelain"],
                                    text=True).strip()
    if branch != repo_branch or dirty:
        raise RuntimeError("Cannot migrate a modified or different branch.")
    subprocess.run(cmd + ["pull", "--ff-only", "origin", repo_branch], check=True)
    if "python_modules.core.repository_sync" in sys.modules:
        raise RuntimeError(
            "Repository updated, but the old sync module is already loaded. "
            "Restart the kernel and rerun from the first cell."
        )

sys.path.insert(0, str(repo_path))
from python_modules.core.repository_sync import sync_repository

repo_path = sync_repository(repo_url, repo_branch, parent=repo_path.parent)
%cd {repo_path}


**01. Imports and Research Environment**

Use standard Colab paths, the pinned MATLAB reader, reusable Essex data recovery and regression utilities, and the shared scientific plotting style.


In [ ]:
# =================== 01. Imports and Research Environment ====================
# -------------------------- 1. Study Environment ----------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("biomechanical_load_estimation")

# -------------------------- 2. Standard Library -----------------------------
import hashlib
import json
from importlib.metadata import version
from pathlib import Path

# -------------------- 3. Scientific Computing and Display -------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
plot_style(font_family="DejaVu Sans", title_size=12.5,
    title_weight="normal", label_size=11.5, label_weight="normal",
    tick_size=11, legend_size=10, figure_dpi=100, figure_size=(9, 4))

# ------------------ 4. Source and Regression Utilities ----------------------
from core.output_download import download_outputs_from_colab
from essex_figshare import (
    ARTICLE_ID, ARTICLE_VERSION, ensure_pinned_matlab_archive,
    inspect_matlab_archive, extract_matlab_files,
)
from essex_numeric_recovery import check_record_registry
from essex_paired_dataset import (
    build_paired_arrays, paired_source_digest, save_ignored_cache,
)
from essex_timebase_audit import EXPECTED_CHANNELS
from essex_regression_baselines import (
    MODELS, TARGETS, trial_features, flatten_targets, make_regressor,
    training_group_folds, regression_metrics, participant_metrics,
    subject_balanced_mae, trial_predictions,
)

# ------------------------- 5. Local Data Locations --------------------------
study_dir = root / "biomechanical_load_estimation"
raw_dir = root / "data" / "raw" / "essex_femoral_neck_loading"
mat_dir = raw_dir / "matlab_records"
processed_dir = root / "data" / "processed" / "essex_femoral_neck_loading"
output_dir = study_dir / "output"
processed_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)
assert output_dir.name == "output"
print("Stage 05 evidence directory:", output_dir)


**02. Pinned Stage 04 Evidence and Protected Split**

Read committed CSV/JSON evidence only. Verify archive identity, the 1,729 eligible trial keys and the original 29/4/4 subject assignment before accessing numerical arrays. The Stage 04 Test partition is not evaluated.


In [ ]:
# ================== 02. Pinned Stage 04 Evidence and Splits ===================
# -------------------------- 1. Load Saved Evidence --------------------------
paths04 = {
    "gate": output_dir / "04_dataset_and_split_quality_gate.json",
    "manifest": output_dir / "04_local_dataset_build_manifest.json",
    "trials": output_dir / "04_trial_split_assignments.csv",
    "subjects": output_dir / "04_subject_split_assignments.csv",
    "source": output_dir / "01_consolidated_source_and_trial_quality_gate.json",
    "registry": output_dir / "01_original_record_registry.csv",
    "mapping": output_dir / "03_original_time_coordinate_mapping.csv",
}
missing04 = [key for key, path in paths04.items() if not path.is_file()]
if missing04:
    raise FileNotFoundError("Missing committed upstream evidence: " +
                            ", ".join(missing04))
gate04 = json.loads(paths04["gate"].read_text(encoding="utf-8"))
manifest04 = json.loads(paths04["manifest"].read_text(encoding="utf-8"))
source01 = json.loads(paths04["source"].read_text(encoding="utf-8"))
assigned = pd.read_csv(paths04["trials"], dtype=str)
subjects04 = pd.read_csv(paths04["subjects"], dtype=str)
assert gate04["stage04_quality_gate"] == "PASSED"
assert gate04["source_article_id"] == ARTICLE_ID == 30225739
assert gate04["source_version"] == ARTICLE_VERSION == 13
assert gate04["original_sha256"] == source01["archive_sha256"]
assert gate04["original_sha256"] == manifest04["original_archive_sha256"]
assert gate04["canonical_pair_sha256"] == manifest04["canonical_pair_sha256"]
assert manifest04["feature_shape"] == [1729, 101, 42]
assert manifest04["target_shape"] == [1729, 11, 2]
assert gate04["original_source_empty_fe"] == 168
assert gate04["original_target_units_as_published"] == "unitless"

# ------------------------ 2. Freeze Participant Roles -----------------------
expected_splits = {"TRAIN": 1354, "VALIDATION": 192, "TEST": 183}
assert assigned["split"].value_counts().to_dict() == expected_splits
assert assigned["local_record_key"].is_unique
assert len(assigned) == 1729
assert subjects04["subject_id"].is_unique
assert subjects04["split"].value_counts().to_dict() == {
    "TRAIN": 29, "VALIDATION": 4, "TEST": 4,
    "EXCLUDED_NO_FE_TARGET": 3,
}
assert assigned.groupby("subject_id")["split"].nunique().eq(1).all()
assert assigned["split"].eq(
    assigned["subject_id"].map(subjects04.set_index("subject_id")["split"])
).all()
train_mask = assigned["split"].eq("TRAIN").to_numpy()
valid_mask = assigned["split"].eq("VALIDATION").to_numpy()
test_mask = assigned["split"].eq("TEST").to_numpy()
assert not np.any(train_mask & valid_mask)
assert not np.any(train_mask & test_mask)
assert not np.any(valid_mask & test_mask)
print("Frozen Training trials:", train_mask.sum())
print("Frozen Validation trials:", valid_mask.sum())
print("Reserved Test trials (no model access):", test_mask.sum())
display(assigned.groupby("split").agg(
    trials=("local_record_key", "size"),
    participants=("subject_id", "nunique"),
))


**03. Verified Local Numerical Cache and Reconstruction**

Reuse the Stage 04 binary cache when available, checking its original record order and canonical SHA256. In a fresh runtime, reconstruct it from checksum-verified original MATLAB records, then recheck the same fingerprint. No historical notebook needs to be re-executed.


In [ ]:
# ============== 03. Verified Cache and Source Reconstruction ================
# ------------------- 1. Pinned Original MATLAB Reader -----------------------
%pip -q install --only-binary=:all: --no-deps "mat-io==1.0.1"
from matio import load_from_mat
assert version("mat-io") == "1.0.1"

# --------------------- 2. Rebuild Cache Only If Missing ----------------------
cache_path = processed_dir / "essex_imu_fe_pairs_v13.npz"
if not cache_path.exists():
    archive, hashes = ensure_pinned_matlab_archive(raw_dir)
    assert hashes["sha256"] == source01["archive_sha256"]
    inventory = inspect_matlab_archive(archive, expected_mat_files=40)
    mat_files = extract_matlab_files(archive, mat_dir, inventory)
    assert len(mat_files) == 40
    registry = pd.read_csv(paths04["registry"], dtype={
        "local_record_key": str, "source_file": str,
        "subject_id": str, "task": str, "updated_trial_id": str,
        "fe_representation": str,
    })
    mapping = pd.read_csv(paths04["mapping"], dtype={
        "local_record_key": str, "imu_position_indices": str,
        "mapping_status": str,
    })
    check_record_registry(registry, source01)
    rebuilt, eligible, excluded = build_paired_arrays(
        load_from_mat, mat_dir, registry, mapping, source01,
    )
    assert len(eligible) == 1729 and len(excluded) == 168
    assert eligible["local_record_key"].tolist() == (
        assigned["local_record_key"].tolist()
    )
    assert paired_source_digest(rebuilt, assigned) == (
        gate04["canonical_pair_sha256"]
    )
    save_ignored_cache(cache_path, rebuilt, assigned)
    del rebuilt, eligible, excluded
    print("Source-verified local cache reconstructed.")
else:
    print("Using existing Stage 04 cache, subject to fingerprint verification.")

# ---------------------- 3. Verify Cached Source Identity --------------------
with np.load(cache_path, allow_pickle=False) as saved:
    assert set(saved.files) == {
        "X_full", "X_at_fe", "y_fe", "imu_positions", "record_keys"
    }
    assert saved["record_keys"].astype(str).tolist() == (
        assigned["local_record_key"].tolist()
    )
    # Original Test source values are included ONLY in this integrity digest;
    # no Test feature/target is supplied to any estimator or metric below.
    source_digest = paired_source_digest(
        {key: saved[key] for key in (
            "X_full", "X_at_fe", "y_fe", "imu_positions"
        )}, assigned,
    )
    assert source_digest == gate04["canonical_pair_sha256"]
    x_train_source = saved["X_full"][train_mask].copy()
    x_valid_source = saved["X_full"][valid_mask].copy()
    y_train_source = saved["y_fe"][train_mask].copy()
    y_valid_source = saved["y_fe"][valid_mask].copy()
assert x_train_source.shape == (1354, 101, 42)
assert x_valid_source.shape == (192, 101, 42)
assert y_train_source.shape == (1354, 11, 2)
assert y_valid_source.shape == (192, 11, 2)
print("Canonical Stage 04 source fingerprint: VERIFIED")
print("Test observations used in modelling or diagnostics: NONE")


**04. Trial-Level Features and Original FE Targets**

Represent each original 101-sample movement trial by five fixed, target-independent statistics for each of 42 IMU channels: mean, population standard deviation, minimum, maximum and RMS (210 predictors). Preserve 11 source FE positions × two targets as 22 outputs per trial.


In [ ]:
# =================== 04. Trial Features and FE Targets =======================
# ---------------------- 1. Extract Fixed IMU Features -----------------------
x_train, feature_names = trial_features(
    x_train_source, EXPECTED_CHANNELS,
)
x_valid, confirmed_names = trial_features(
    x_valid_source, EXPECTED_CHANNELS,
)
assert confirmed_names == feature_names and len(feature_names) == 210
y_train = y_train_source
y_valid = y_valid_source
z_train = flatten_targets(y_train)
z_valid = flatten_targets(y_valid)
assert x_train.shape == (1354, 210) and x_valid.shape == (192, 210)
assert z_train.shape == (1354, 22) and z_valid.shape == (192, 22)

# ---------------------- 2. Preserve Original Trial IDs ----------------------
train_info = assigned.loc[train_mask].reset_index(drop=True)
valid_info = assigned.loc[valid_mask].reset_index(drop=True)
assert not set(train_info["subject_id"]) & set(valid_info["subject_id"])
assert not set(train_info["subject_id"]) & set(
    assigned.loc[test_mask, "subject_id"]
)
feature_schema = pd.DataFrame({
    "feature_index": np.arange(210),
    "feature": feature_names,
    "statistic": np.repeat(
        ["mean", "std", "minimum", "maximum", "rms"], 42,
    ),
    "original_channel": list(EXPECTED_CHANNELS) * 5,
})
feature_schema.to_csv(
    output_dir / "05_feature_schema.csv", index=False,
)
print("Original trials as learning rows:", len(x_train))
print("Predictors:", len(feature_names), "| FE outputs per trial:", 22)
display(feature_schema.head(10))


**05. Pre-Specified Models and Training-Only Grouped Cross-Validation**

Use one fixed five-fold GroupKFold over the 29 Training participants. Each fold fits its own preprocessing, including a StandardScaler inside Ridge only. The three model configurations are fixed before any Validation scoring. Do not use Test to determine folds, parameters or predictions.


In [ ]:
# =============== 05. Training-Only Grouped Cross-Validation ================
# --------------------- 1. Fix Participant-Disjoint Folds ---------------------
CV_FOLDS = 5
SEED = 2026
train_groups = train_info["subject_id"].to_numpy()
folds, fold_ids = training_group_folds(
    train_groups, n_splits=CV_FOLDS,
)
fold_registry = train_info.loc[:, [
    "local_record_key", "subject_id", "task"
]].copy()
fold_registry["validation_fold"] = fold_ids
fold_registry.to_csv(
    output_dir / "05_training_fold_registry.csv", index=False,
)
assert set(fold_registry["subject_id"]).isdisjoint(
    set(valid_info["subject_id"])
)
assert set(fold_registry["subject_id"]).isdisjoint(
    set(assigned.loc[test_mask, "subject_id"])
)

# ---------------- 2. Fit Fresh Estimators in Every Training Fold -------------
cv_rows = []
oof = {}
for name in MODELS:
    estimates = np.full(y_train.shape, np.nan, dtype=np.float64)
    for fold_number, (fit_rows, held_rows) in enumerate(folds, 1):
        learner = make_regressor(name, seed=SEED)
        learner.fit(x_train[fit_rows], z_train[fit_rows])
        pred = learner.predict(x_train[held_rows]).reshape(-1, 11, 2)
        estimates[held_rows] = pred
        assert not set(train_groups[fit_rows]) & set(
            train_groups[held_rows]
        )
        fold_score = subject_balanced_mae(
            y_train[held_rows], pred, train_groups[held_rows],
        )
        for result in regression_metrics(y_train[held_rows], pred):
            cv_rows.append({
                "model": name, "fold": fold_number,
                "training_subjects": len(set(train_groups[fit_rows])),
                "validation_subjects": len(set(train_groups[held_rows])),
                "subject_balanced_mae": fold_score, **result,
            })
    assert np.isfinite(estimates).all()
    oof[name] = estimates
cv_folds = pd.DataFrame(cv_rows)
assert len(cv_folds) == len(MODELS) * CV_FOLDS * 2
cv_folds.to_csv(
    output_dir / "05_training_cv_fold_metrics.csv", index=False,
)
print("Completed Training-only grouped folds:", CV_FOLDS)
display(cv_folds.head(10))


**06. Cross-Validated Participant and Activity Errors**

Score every Training subject only from their out-of-fold predictions. Keep errors for e1 and e3 separate, report the subject-balanced score and choose the candidate model by Training CV alone.


In [ ]:
# ============ 06. Training Out-of-Fold Error and Model Choice ===============
# ----------------- 1. Source-Linked Participant Diagnostics -----------------
cv_person_tables = []
summary_rows = []
for name in MODELS:
    pred = oof[name]
    rows = participant_metrics(
        y_train, pred, train_info["subject_id"], train_info["task"],
        model=name, phase="TRAIN_OOF",
    )
    rows["fold"] = rows["subject_id"].map(
        fold_registry.drop_duplicates("subject_id").set_index(
            "subject_id"
        )["validation_fold"]
    ).astype(int)
    cv_person_tables.append(rows)
    score = subject_balanced_mae(y_train, pred, train_groups)
    summary_rows.append({
        "model": name,
        "subject_balanced_mae": score,
        "e1_oof_mae": regression_metrics(y_train, pred)[0]["mae"],
        "e3_oof_mae": regression_metrics(y_train, pred)[1]["mae"],
        "training_subjects": len(set(train_groups)),
        "training_trials": len(y_train),
        "cv_folds": CV_FOLDS,
    })
cv_person = pd.concat(cv_person_tables, ignore_index=True)
cv_summary = pd.DataFrame(summary_rows).sort_values(
    ["subject_balanced_mae", "model"], kind="stable",
).reset_index(drop=True)
selected_model = str(cv_summary.loc[0, "model"])
cv_person.to_csv(
    output_dir / "05_training_cv_subject_activity_metrics.csv",
    index=False,
)
cv_summary.to_csv(
    output_dir / "05_training_cv_model_summary.csv",
    index=False,
)
assert selected_model in MODELS
print("Selected on Training GroupKFold only:", selected_model)
display(cv_summary)


**07. Frozen Validation Evaluation**

After Training-only model selection, refit each of the three pre-specified models on the complete Training split. Evaluate all candidates on the same four Validation participants for transparent diagnostics; do not change the selected model based on these results.


In [ ]:
# ==================== 07. Frozen Validation Evaluation ======================
# ------------------ 1. Train on the Frozen Training Split -------------------
valid_predictions = {}
validation_metrics_rows = []
validation_long = []
for name in MODELS:
    learner = make_regressor(name, seed=SEED)
    learner.fit(x_train, z_train)
    prediction = learner.predict(x_valid).reshape(-1, 11, 2)
    assert np.isfinite(prediction).all()
    valid_predictions[name] = prediction
    balanced = subject_balanced_mae(
        y_valid, prediction, valid_info["subject_id"],
    )
    for item in regression_metrics(y_valid, prediction):
        validation_metrics_rows.append({
            "model": name, "selected_from_training_cv": name == selected_model,
            "subject_balanced_mae": balanced,
            "validation_participants": 4,
            **item,
        })
    validation_long.append(trial_predictions(
        y_valid, prediction,
        valid_info["local_record_key"], valid_info["subject_id"],
        valid_info["task"], name,
    ))
validation_metrics = pd.DataFrame(validation_metrics_rows)
validation_predictions = pd.concat(validation_long, ignore_index=True)
assert len(validation_predictions) == 192 * 11 * 2 * len(MODELS)
assert not set(validation_predictions["subject_id"]) & set(
    assigned.loc[test_mask, "subject_id"]
)
validation_metrics.to_csv(
    output_dir / "05_validation_model_metrics.csv", index=False,
)
validation_predictions.to_csv(
    output_dir / "05_validation_predictions.csv", index=False,
)
print("Training-CV-selected model:", selected_model)
print("Validation trials per candidate:", len(y_valid))
display(validation_metrics)


**08. Participant, Activity and Biomechanical Peak Analysis**

Report participant/activity heterogeneity without selecting models from Validation. MAE and RMSE aggregate error over the original FE coordinates; peak-value and peak-position errors use 11 source-grid positions, not interpolated time or external physiological calibration.


In [ ]:
# ============== 08. Participant, Activity and Peak Analysis ================
# ----------------- 1. Person- and Movement-Level Results --------------------
validation_details = pd.concat([
    participant_metrics(
        y_valid, valid_predictions[name],
        valid_info["subject_id"], valid_info["task"],
        model=name, phase="VALIDATION",
    ) for name in MODELS
], ignore_index=True)
assert validation_details["subject_id"].nunique() == 4
assert set(validation_details["target"]) == set(TARGETS)
validation_details.to_csv(
    output_dir / "05_validation_subject_activity_metrics.csv",
    index=False,
)
activity_report = validation_details.groupby(
    ["model", "task", "target"], as_index=False
).agg(
    mean_subject_activity_mae=("mae", "mean"),
    participants=("subject_id", "nunique"),
    trials=("trials", "sum"),
)
print("Validation error by movement and FE target:")
display(activity_report)
print("Peak index error uses native FE positions 0 to 10 only.")


**09. Scientific Comparison and Original-Grid Waveforms**

Create a grouped MAE comparison plot and two genuine FE waveform examples. Select the illustrated Validation trial by its source key rather than by prediction error; label it as an example, not a representative performance claim.


In [ ]:
# ====================== 09. Scientific Result Figures ========================
# ----------------- 1. Side-by-Side Validation MAE by Model ------------------
chart = validation_metrics.pivot(
    index="model", columns="target", values="mae",
).reindex(MODELS)
fig, ax = plt.subplots(figsize=(9, 4))
chart.plot.bar(ax=ax, width=0.75)
ax.set_title("Validation MAE by regression baseline and FE target")
ax.set_xlabel("Regression model")
ax.set_ylabel("MAE (published unitless FE scale)")
ax.tick_params(axis="x", rotation=15)
save_figure(fig, output_dir / "05_validation_model_comparison.png")
plt.show()

# ---------------- 2. Fixed, Source-Identified Waveform Demonstrations ---------
example_index = int(np.argsort(
    valid_info["local_record_key"].to_numpy()
)[0])
example_key = str(valid_info.iloc[example_index]["local_record_key"])
print("Illustrative Validation trial:", example_key)
predicted = valid_predictions[selected_model][example_index]
observed = y_valid[example_index]
for j, target in enumerate(TARGETS):
    fig, ax = plt.subplots(figsize=(9, 4))
    position = np.arange(11)
    ax.plot(position, observed[:, j], marker="o", label="Source FE")
    ax.plot(position, predicted[:, j], marker="s",
            label=f"{selected_model} prediction")
    ax.set_title(f"Validation example: {target} | {example_key}")
    ax.set_xlabel("Original FE position (index 0–10)")
    ax.set_ylabel("Published unitless FE strain")
    ax.set_xticks(position)
    ax.legend()
    save_figure(
        fig, output_dir / f"05_validation_waveform_{target}.png",
    )
    plt.show()


**10. Provenance, Quality Gate and Output Packaging**

Record actual model settings, upstream file hashes, split identity and completed evaluation checks. The Quality Gate only passes after the real Training and Validation runs. Bundle the public, flat `05_*` outputs for manual upload; no private numeric array or trained estimator is committed.


In [ ]:
# ================= 10. Provenance and Research Quality Gate =================
# ------------------------ 1. Audit Actual Evidence --------------------------
source_sha = {
    name: hashlib.sha256(path.read_bytes()).hexdigest()
    for name, path in paths04.items()
}
assert selected_model == cv_summary.loc[0, "model"]
assert len(folds) == 5
assert len(set(train_info["subject_id"])) == 29
assert len(set(valid_info["subject_id"])) == 4
assert len(set(assigned.loc[test_mask, "subject_id"])) == 4
assert len(set(train_info["subject_id"]) & set(valid_info["subject_id"])) == 0
assert set(train_info["subject_id"]).isdisjoint(
    set(assigned.loc[test_mask, "subject_id"])
)
assert set(valid_info["subject_id"]).isdisjoint(
    set(assigned.loc[test_mask, "subject_id"])
)
assert (cv_summary["subject_balanced_mae"] >= 0).all()
assert (validation_metrics["mae"] >= 0).all()
assert (validation_metrics["rmse"] >= 0).all()
assert np.isfinite(validation_predictions["predicted"]).all()

# ------------------- 2. Save a Source-Bounded Quality Record -----------------
protocol = {
    "stage": "05",
    "quality_gate": "PASSED",
    "source_article_id": ARTICLE_ID,
    "source_version": ARTICLE_VERSION,
    "source_archive_sha256": gate04["original_sha256"],
    "stage04_canonical_pair_sha256": source_digest,
    "upstream_file_sha256": source_sha,
    "training_participants": 29,
    "training_trials": 1354,
    "validation_participants": 4,
    "validation_trials": 192,
    "reserved_test_participants": 4,
    "reserved_test_trials": 183,
    "test_model_fitting": False,
    "test_model_selection": False,
    "test_performance_evaluated": False,
    "train_group_cv_folds": CV_FOLDS,
    "selection_metric": "equal-subject-weight mean of e1/e3 OOF MAE",
    "selected_by_training_cv": selected_model,
    "predeclared_models": {
        "Training Mean": {"strategy": "mean"},
        "Ridge": {"alpha": 10.0, "scaler": "fold_training_only"},
        "Random Forest": {
            "n_estimators": 200, "min_samples_leaf": 3,
            "max_features": 1.0, "seed": SEED,
        },
    },
    "feature_contract": "five statistics per 42 original IMU channels",
    "feature_count": 210,
    "prediction_values_per_trial": 22,
    "fe_positions": 11,
    "targets": list(TARGETS),
    "published_target_unit": "unitless",
    "fe_interpolation": False,
    "clinical_or_external_validation": False,
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "scikit_learn_version": version("scikit-learn"),
}
expected05 = [
    "05_feature_schema.csv",
    "05_training_fold_registry.csv",
    "05_training_cv_fold_metrics.csv",
    "05_training_cv_subject_activity_metrics.csv",
    "05_training_cv_model_summary.csv",
    "05_validation_model_metrics.csv",
    "05_validation_predictions.csv",
    "05_validation_subject_activity_metrics.csv",
    "05_validation_model_comparison.png",
    "05_validation_waveform_e1.png",
    "05_validation_waveform_e3.png",
    "05_regression_protocol_and_quality_gate.json",
]
gate_path = output_dir / "05_regression_protocol_and_quality_gate.json"
gate_path.write_text(json.dumps(protocol, indent=2) + "\n",
                     encoding="utf-8")
missing05 = [filename for filename in expected05
             if not (output_dir / filename).is_file()]
if missing05:
    raise FileNotFoundError("Incomplete Stage 05 outputs: " +
                            ", ".join(missing05))
print("STAGE 05 | TRAINING-ONLY BASELINES AND VALIDATION GATE: PASSED")
print("Validated Stage 05 public evidence files:", len(expected05))
download_outputs_from_colab(
    folder="biomechanical_load_estimation/output", stage="05",
)


**11. Scientific Interpretation and Next Research Boundary**

This notebook establishes three pre-specified, trial-wise, multi-output regression baselines for the original Essex FE-model `e1` and `e3` curves. Training-only grouped CV supports model choice; the independent, predefined Validation subjects provide diagnostics. The eleven source FE coordinates are never treated as independent learning trials. Report pointwise MAE/RMSE, participant and activity heterogeneity, peak-value error, and peak-location error on the original 11-position grid.

**Interpreting results:** Any numerical scores must come from execution of the preceding cells. Comparing Validation model scores is descriptive and does not retrospectively change the Training-CV-selected model. Between-trial variation, participant imbalance and the very small four-person Validation cohort limit precision and transferability claims.

**Next research boundary:** Proceed to carefully budgeted temporal deep learning or additional robustness experiments *using Training/Validation only*. Preserve all four untouched Test participants for a separately approved final model freeze and qualified assessment. No physical FE calibration, prospective application, clinical validation or independent external generalisation is established.
